# Seeded Run 1 — All-Class (20-way SHD) Task

This notebook trains five matched seeded pairs for the full 20-class SHD classification task.

Each training cell handles one seed pair independently. Once a pair finishes, its checkpoints and summaries are already saved, so you can stop there and later continue with the next seed cell instead of rerunning a full loop.

Cell 2 defines the seeds used in this notebook. Cell 3 verifies three things before training starts:
1. this notebook is locked to the 20-class (all speakers) SHD classification task
2. the homogeneous and heterogeneous model definitions are set up correctly
3. the log-normal hidden tau_m samples vary across the configured seeds

In [1]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1"
TASK_KEY = "allclass"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]

print(f"Device: {DEVICE}")
print(f"Train file exists: {SHD_TRAIN.exists()}")
print(f"Test file exists: {SHD_TEST.exists()}")
print(f"Tau artifact exists: {TAU_ARTIFACT_PATH.exists()}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")
print(f"Task outputs: {TASK_DEF['nb_outputs']}")

Device: cuda
Train file exists: True
Test file exists: True
Tau artifact exists: True
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1\allclass_lognormal
Master seeds: [101, 202, 210, 340, 440]
Task name: all_class_shd
Task outputs: 20


In [2]:
assert TASK_KEY == "allclass"
assert TASK_DEF["nb_outputs"] == 20
assert TASK_DEF["task_name"] == "all_class_shd"

# Verify all-class label map: identity mapping (speaker i → class i)
expected_allclass_map = {i: i for i in range(20)}
assert TASK_DEF["task_label_map"] == expected_allclass_map

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 20
assert preview["hetero_prms"]["nb_outputs"] == 20
assert not preview["hom_model"].network[0].alpha.requires_grad
assert not preview["hom_model"].network[0].beta.requires_grad
assert not preview["hetero_model"].network[0].alpha.requires_grad
assert not preview["hetero_model"].network[0].beta.requires_grad
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)

assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("All-class (20-way SHD) task mapping verified.")
print("Homogeneous anchor and heterogeneous sampled definitions verified.")
print("Log-normal hidden tau_m sampling varies across the configured seeds.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,101,allclass,all_class_shd,lognormal,True,30,1,24.687252,3.410472,99.749887,37.499115,30.195999,False
1,202,allclass,all_class_shd,lognormal,True,28,1,29.308363,6.953492,99.749887,39.884979,30.480470,False
2,210,allclass,all_class_shd,lognormal,True,31,1,22.405558,3.855315,99.749887,34.397371,29.713145,False
3,340,allclass,all_class_shd,lognormal,True,31,1,23.273864,4.656051,99.749887,35.008043,30.389582,False
4,440,allclass,all_class_shd,lognormal,True,28,1,28.874491,5.547958,99.749887,41.448743,31.915950,False


All-class (20-way SHD) task mapping verified.
Homogeneous anchor and heterogeneous sampled definitions verified.
Log-normal hidden tau_m sampling varies across the configured seeds.


In [3]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [14]:
# ── All-Class Training: Convergence + Speed Optimizations ─────────
#   1. nb_epochs 35→60: train acc still climbing, need more time
#   2. lr 4e-3→2e-3: less aggressive, reduces late-epoch instability
#   3. lr_patience 7→4: cut LR sooner when test loss plateaus
#   4. batch_size 256→512: halves iterations/epoch
#   5. weight_decay 1e-4: regularization
#   6. cuDNN benchmark: auto-tune conv kernels

import torch
import importlib

# Reload common module to pick up allclass task + LR scheduler changes
seeded_runs_common = importlib.reload(seeded_runs_common)

# Re-bind all names
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

TASK_DEF = TASKS[TASK_KEY]

# ── Speed settings ──
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

seeded_runs_common.BASE_PRMS["batch_size"] = 512          # was 256: halves iterations
seeded_runs_common.BASE_PRMS["nb_epochs"] = 60            # was 35→60: train acc still climbing
seeded_runs_common.BASE_PRMS["compile_model"] = False     # Triton not available — skip compile

# ── Convergence settings ──
seeded_runs_common.BASE_PRMS["lr"] = 2e-3                 # was 4e-3: less aggressive
seeded_runs_common.BASE_PRMS["lr_ab"] = 2e-3              # was 4e-3: match LR for alpha/beta
seeded_runs_common.BASE_PRMS["weight_decay"] = 1e-4       # was 0.0: regularize
seeded_runs_common.BASE_PRMS["lr_patience"] = 4           # was 7→4: cut LR sooner
seeded_runs_common.BASE_PRMS["lr_factor"] = 0.5           # halve LR on plateau
seeded_runs_common.BASE_PRMS["lr_min"] = 5e-5             # scheduler floor

print("All-class convergence + speed optimizations applied:")
print(f"  nb_epochs      = {seeded_runs_common.BASE_PRMS['nb_epochs']}   (was 35)")
print(f"  lr / lr_ab     = {seeded_runs_common.BASE_PRMS['lr']:.0e}  (was 4e-3)")
print(f"  batch_size     = {seeded_runs_common.BASE_PRMS['batch_size']}")
print(f"  weight_decay   = {seeded_runs_common.BASE_PRMS['weight_decay']}")
print(f"  LR scheduler   = ReduceLROnPlateau(patience=4, factor=0.5)")
print(f"  compile_model  = {seeded_runs_common.BASE_PRMS['compile_model']}")
print(f"  cuDNN benchmark = {torch.backends.cudnn.benchmark if torch.cuda.is_available() else 'N/A'}")
print(f"  Est. time/seed: ~85 min (60 epochs × ~1.4 min)")


All-class speed + convergence optimizations applied:
  batch_size     = 512  (was 256)
  nb_epochs      = 35   (was 25)
  compile_model  = False
  weight_decay   = 0.0001
  LR scheduler   = ReduceLROnPlateau(patience=7, factor=0.5)
  cuDNN benchmark = True
  Estimated speedup: ~2.5× (batch_size + epochs + compile)
  Estimated time/seed: ~2h (was ~5h)


In [12]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)

    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)

    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index()
            .sort_values("pair_seed")
            .reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)

    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed,
        train_cache=train_cache,
        test_cache=test_cache,
        task_key=TASK_KEY,
        mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL,
        skip_existing=True,
    )

    checkpoint_rows = upsert_rows(
        read_manifest_rows(RESULT_STEM),
        run_rows,
        CHECKPOINT_KEY_FIELDS,
    )
    pair_rows = upsert_rows(
        read_manifest_rows(PAIR_STEM),
        [build_pair_summary_row(pair_meta)],
        PAIR_KEY_FIELDS,
    )

    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)

    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Run helpers ready.")
print("Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.")
print("Chance level for 20-class task: 5.0%")

Run helpers ready.
Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.
Chance level for 20-class task: 5.0%


In [9]:
# Train or reuse seed pair 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(101)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 101].reset_index(drop=True))


Seed 101 :: reusing existing homogeneous_anchor checkpoint...

Seed 101 :: reusing existing heterogeneous_sampled checkpoint...


,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,101,heterogeneous_sampled,0.424912,1.529589,9265.432745
1,101,homogeneous_anchor,0.583481,1.213639,9195.480824


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,allclass,all_class_shd,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6


In [13]:
# Train or reuse seed pair 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(202)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 202].reset_index(drop=True))


Seed 202 :: training homogeneous_anchor...
  torch.compile applied (reduce-overhead mode)


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
W0608 23:00:30.404000 19452 site-packages\torch\_dynamo\convert_frame.py:1233] WON'T CONVERT forward C:\Users\Priya\Desktop\research project (SNN Info Theory)\neural_heterogeneity\SuGD_code\SuSpike.py line 15 
W0608 23:00:30.404000 19452 site-packages\torch\_dynamo\convert_frame.py:1233] due to: 
W0608 23:00:30.404000 19452 site-packages\torch\_dynamo\convert_frame.py:1233] Traceback (most recent call last):
W0608 23:00:30.404000 19452 site-packages\torch\_dynamo\convert_frame.py:1233]   File "c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\convert_frame.py", line 1164, in __call__
W0608 23:00:30.404000 19452 site-packages\torch\_dynamo\convert_frame.py:1233]     result = self._inner_convert(
W0608 23:00:30.404000 19452 site-

  epoch=001  train_acc=0.074  test_acc=0.105  (179.4 min)
  epoch=002  train_acc=0.107  test_acc=0.133  (1.4 min)
  epoch=003  train_acc=0.158  test_acc=0.198  (1.5 min)
  epoch=004  train_acc=0.221  test_acc=0.266  (1.4 min)
  epoch=005  train_acc=0.245  test_acc=0.270  (1.4 min)
  epoch=006  train_acc=0.272  test_acc=0.320  (1.4 min)
  epoch=007  train_acc=0.300  test_acc=0.322  (1.4 min)
  epoch=008  train_acc=0.319  test_acc=0.385  (1.4 min)
  epoch=009  train_acc=0.343  test_acc=0.402  (1.4 min)
  epoch=010  train_acc=0.354  test_acc=0.421  (1.5 min)
  epoch=011  train_acc=0.369  test_acc=0.408  (1.5 min)
  epoch=012  train_acc=0.382  test_acc=0.433  (1.5 min)
  epoch=013  train_acc=0.391  test_acc=0.372  (1.5 min)
  epoch=014  train_acc=0.397  test_acc=0.430  (1.5 min)
  epoch=015  train_acc=0.421  test_acc=0.457  (1.5 min)
  epoch=016  train_acc=0.441  test_acc=0.439  (1.5 min)
  epoch=017  train_acc=0.449  test_acc=0.481  (1.5 min)
  epoch=018  train_acc=0.450  test_acc=0.492  

c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.084  test_acc=0.136  (1.4 min)
  epoch=002  train_acc=0.125  test_acc=0.152  (1.4 min)
  epoch=003  train_acc=0.162  test_acc=0.229  (1.4 min)
  epoch=004  train_acc=0.221  test_acc=0.271  (1.4 min)
  epoch=005  train_acc=0.256  test_acc=0.258  (1.4 min)
  epoch=006  train_acc=0.282  test_acc=0.317  (1.4 min)
  epoch=007  train_acc=0.314  test_acc=0.326  (1.4 min)
  epoch=008  train_acc=0.346  test_acc=0.359  (1.4 min)
  epoch=009  train_acc=0.356  test_acc=0.366  (1.5 min)
  epoch=010  train_acc=0.373  test_acc=0.385  (1.5 min)
  epoch=011  train_acc=0.388  test_acc=0.377  (1.5 min)
  epoch=012  train_acc=0.400  test_acc=0.399  (1.5 min)
  epoch=013  train_acc=0.417  test_acc=0.385  (1.5 min)
  epoch=014  train_acc=0.401  test_acc=0.415  (1.5 min)
  epoch=015  train_acc=0.421  test_acc=0.413  (1.5 min)
  epoch=016  train_acc=0.431  test_acc=0.393  (1.6 min)
  epoch=017  train_acc=0.443  test_acc=0.405  (1.6 min)
  epoch=018  train_acc=0.443  test_acc=0.428  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,202,heterogeneous_sampled,0.492933,1.456393,3045.233922
1,202,homogeneous_anchor,0.504859,1.410907,13703.839932


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,202,allclass,all_class_shd,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.48047,28,1,True,6


In [15]:
# Train or reuse seed pair 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(210)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 210].reset_index(drop=True))


Seed 210 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.107  test_acc=0.169  (1.4 min)
  epoch=002  train_acc=0.157  test_acc=0.203  (1.4 min)
  epoch=003  train_acc=0.202  test_acc=0.234  (1.4 min)
  epoch=004  train_acc=0.256  test_acc=0.273  (1.4 min)
  epoch=005  train_acc=0.300  test_acc=0.340  (1.4 min)
  epoch=006  train_acc=0.332  test_acc=0.360  (1.4 min)
  epoch=007  train_acc=0.361  test_acc=0.399  (1.5 min)
  epoch=008  train_acc=0.390  test_acc=0.347  (1.5 min)
  epoch=009  train_acc=0.396  test_acc=0.381  (1.5 min)
  epoch=010  train_acc=0.419  test_acc=0.395  (1.5 min)
  epoch=011  train_acc=0.439  test_acc=0.427  (1.5 min)
  epoch=012  train_acc=0.447  test_acc=0.430  (1.5 min)
  epoch=013  train_acc=0.456  test_acc=0.439  (1.5 min)
  epoch=014  train_acc=0.455  test_acc=0.466  (1.5 min)
  epoch=015  train_acc=0.486  test_acc=0.473  (1.6 min)
  epoch=016  train_acc=0.496  test_acc=0.486  (1.5 min)
  epoch=017  train_acc=0.503  test_acc=0.499  (1.5 min)
  epoch=018  train_acc=0.502  test_acc=0.464  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,210,heterogeneous_sampled,0.475265,1.510586,3021.519277
1,210,homogeneous_anchor,0.574647,1.241746,3037.916760


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,210,allclass,all_class_shd,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6


In [16]:
# Train or reuse seed pair 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(340)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 340].reset_index(drop=True))


Seed 340 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.091  test_acc=0.167  (1.4 min)
  epoch=002  train_acc=0.166  test_acc=0.199  (1.4 min)
  epoch=003  train_acc=0.218  test_acc=0.249  (1.5 min)
  epoch=004  train_acc=0.258  test_acc=0.326  (1.5 min)
  epoch=005  train_acc=0.293  test_acc=0.318  (1.5 min)
  epoch=006  train_acc=0.325  test_acc=0.371  (1.4 min)
  epoch=007  train_acc=0.353  test_acc=0.367  (1.5 min)
  epoch=008  train_acc=0.371  test_acc=0.388  (1.5 min)
  epoch=009  train_acc=0.394  test_acc=0.394  (1.5 min)
  epoch=010  train_acc=0.418  test_acc=0.427  (1.5 min)
  epoch=011  train_acc=0.429  test_acc=0.445  (1.5 min)
  epoch=012  train_acc=0.445  test_acc=0.425  (1.5 min)
  epoch=013  train_acc=0.446  test_acc=0.413  (1.5 min)
  epoch=014  train_acc=0.468  test_acc=0.423  (1.6 min)
  epoch=015  train_acc=0.483  test_acc=0.444  (1.5 min)
  epoch=016  train_acc=0.498  test_acc=0.447  (1.6 min)
  epoch=017  train_acc=0.500  test_acc=0.487  (1.4 min)
  epoch=018  train_acc=0.501  test_acc=0.491  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,340,heterogeneous_sampled,0.431095,1.561801,3033.600237
1,340,homogeneous_anchor,0.525618,1.308563,3040.105785


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,340,allclass,all_class_shd,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6


In [17]:
# Train or reuse seed pair 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(440)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 440].reset_index(drop=True))


Seed 440 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.084  test_acc=0.126  (1.5 min)
  epoch=002  train_acc=0.170  test_acc=0.215  (1.5 min)
  epoch=003  train_acc=0.232  test_acc=0.255  (1.4 min)
  epoch=004  train_acc=0.250  test_acc=0.303  (1.5 min)
  epoch=005  train_acc=0.297  test_acc=0.328  (1.5 min)
  epoch=006  train_acc=0.320  test_acc=0.370  (1.5 min)
  epoch=007  train_acc=0.335  test_acc=0.387  (1.5 min)
  epoch=008  train_acc=0.345  test_acc=0.388  (1.5 min)
  epoch=009  train_acc=0.372  test_acc=0.396  (1.5 min)
  epoch=010  train_acc=0.392  test_acc=0.417  (1.5 min)
  epoch=011  train_acc=0.397  test_acc=0.396  (1.5 min)
  epoch=012  train_acc=0.415  test_acc=0.419  (1.5 min)
  epoch=013  train_acc=0.429  test_acc=0.432  (1.6 min)
  epoch=014  train_acc=0.438  test_acc=0.419  (1.6 min)
  epoch=015  train_acc=0.436  test_acc=0.420  (1.6 min)
  epoch=016  train_acc=0.450  test_acc=0.451  (1.4 min)
  epoch=017  train_acc=0.469  test_acc=0.428  (1.4 min)
  epoch=018  train_acc=0.469  test_acc=0.459  (1

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,440,heterogeneous_sampled,0.488516,1.474193,3032.910258
1,440,homogeneous_anchor,0.432420,1.656537,3022.414432


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,440,allclass,all_class_shd,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.91595,28,1,True,6


In [18]:
# Final status summary
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if pair_df.empty:
    print("No saved seed summaries yet.")
else:
    display(pair_df)

if checkpoint_df.empty:
    print("No saved checkpoint summaries yet.")
else:
    display(checkpoint_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss"]])

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved paired accuracy view to: {PAIRED_ACC_CSV}")
    
    # Quick heterogeneity assessment
    mean_delta = paired_acc_df["hetero_minus_hom"].mean()
    print(f"\nMean hetero − homo Δacc: {mean_delta*100:+.2f} pp")
    if mean_delta >= 0.03:
        print("✓ Heterogeneity improvement ≥ 3 pp — matches or exceeds Perez-Nieves benchmark")
    elif mean_delta >= 0.02:
        print("~ Heterogeneity improvement ≥ 2 pp — modest but consistent advantage")
    elif mean_delta >= 0:
        print("~ Slight positive trend — more seeds may clarify")
    else:
        print("✗ No heterogeneity advantage detected in this run")

,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,allclass,all_class_shd,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6
1,202,allclass,all_class_shd,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.480470,28,1,True,6
2,210,allclass,all_class_shd,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6
3,340,allclass,all_class_shd,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6
4,440,allclass,all_class_shd,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.915950,28,1,True,6


,pair_seed,pair_role,final_test_acc,final_test_loss
0,101,heterogeneous_sampled,0.424912,1.529589
1,101,homogeneous_anchor,0.583481,1.213639
2,202,heterogeneous_sampled,0.492933,1.456393
3,202,homogeneous_anchor,0.504859,1.410907
4,210,heterogeneous_sampled,0.475265,1.510586
5,210,homogeneous_anchor,0.574647,1.241746
6,340,heterogeneous_sampled,0.431095,1.561801
7,340,homogeneous_anchor,0.525618,1.308563
8,440,heterogeneous_sampled,0.488516,1.474193
9,440,homogeneous_anchor,0.432420,1.656537


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom
0,101,0.424912,0.583481,-0.158569
1,202,0.492933,0.504859,-0.011926
2,210,0.475265,0.574647,-0.099382
3,340,0.431095,0.525618,-0.094523
4,440,0.488516,0.432420,0.056095


Saved paired accuracy view to: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1\allclass_lognormal\seeded_run_1_paired_accuracy.csv

Mean hetero − homo Δacc: -6.17 pp
✗ No heterogeneity advantage detected in this run
